# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafsa-SE/flyrank-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue combines two things, not one: **risk** (the Week-5/6 Random Forest's probability
that a page is declining, validated on a client-grouped held-out split at precision@50 = 0.78)
and **value** (`clicks_90d x cpc`, the paper's own "captured click-equivalent value" proxy from
Finding #9 -- not `impressions x cpc`, which the paper flags as unsafe/inflated). Priority
favors pages that are both likely declining AND already earning real value, since that's where
a fix has the most to protect.

**Archetypes -> action mapping** (a simplified version of the paper's age/freshness read,
Findings #2, #4, #8): I bucket each page on two axes -- position (visible / not) and freshness
(fresh / aging / stale) -- into five archetypes, each with a default action:

| Archetype | Definition (freshness/CTR-based, not the model score) | Default action |
|---|---|---|
| `protect_and_expand` | Visible, fresh (<90d), CTR at/above its tier benchmark | Leave layout alone; consider expanding thin sections |
| `refresh_priority` | Visible, stale (90+ days untouched) | Refresh content -- ties to the paper's Finding #4 decay/refresh insight |
| `ctr_fix_candidate` | Visible, fresh, but CTR below tier benchmark | Fix title/snippet, not full content |
| `monitor_only` | Not yet visible enough to score reliably (low impressions) | Watch; no action yet -- too little signal |
| `low_priority_deep` | Deep/page_3-5 position, fresh, CTR ok, but low visibility tier | Lowest priority; review only if capacity allows |

The archetype is a rules-based bucket (freshness + CTR-vs-benchmark), independent of the
model. The Random Forest's `risk_score` is a **separate** signal layered on top for ranking
*within* an archetype -- so a `protect_and_expand` row can still carry a high `risk_score`
(worth a second look even though its CTR/freshness look fine), and that's intentional: the
two signals catch different things.

**Reason codes** on every row explain *why* it landed where it did (e.g.
`high_risk_visible_stale`, `ctr_gap_visible_fresh`, `low_signal_not_visible`) so a reviewer
never has to trust the score blindly.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.width", 130)

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
NUMERIC_FEATURES += ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"]

numeric_frame = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[CATEGORICAL_FEATURES].fillna("unknown").astype(str)
encoded = pd.get_dummies(categorical_frame, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

# This is the DEPLOYED scoring model -- trained on all available rows, the normal way to ship
# a final model once validation is done. Its trustworthy performance number is NOT re-measured
# here: it's the client-grouped precision@50 = 0.78 already validated in w05/w06 on held-out
# clients. Training on everything now just gives the widest possible coverage for the queue.
RANDOM_STATE = 42
final_model = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                      class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)
final_model.fit(X, y)
risk_score = final_model.predict_proba(X)[:, 1]

# --- Value proxy: clicks x cpc (paper Finding #9's defensible formula, NOT impressions x cpc) ---
# cpc has ~2.5K missing values (no benchmark keyword data) -- treat as 0 value contribution
# rather than dropping the row, so those pages still get scored on risk alone.
value_score = (df["clicks_90d"].fillna(0) * df["cpc"].fillna(0)).clip(lower=0)

# --- Archetypes ---
is_visible = (df["avg_position"] > 0) & (df["impressions_90d"] >= 500)
is_stale = df["days_since_last_update"] >= 90
measurable = df[is_visible]
tier_benchmark_ctr = measurable.groupby("position_tier", observed=True)["ctr"].median()
ctr_below_benchmark = df["ctr"] < df["position_tier"].map(tier_benchmark_ctr)

archetype = np.select(
    [
        ~is_visible,
        is_visible & is_stale,
        is_visible & ~is_stale & ctr_below_benchmark,
        is_visible & ~is_stale & ~ctr_below_benchmark & (df["position_tier"].isin(["deep", "page_3_5"])),
        is_visible & ~is_stale & ~ctr_below_benchmark,
    ],
    ["monitor_only", "refresh_priority", "ctr_fix_candidate", "low_priority_deep", "protect_and_expand"],
    default="monitor_only",
)
df["archetype"] = archetype

ACTION_BY_ARCHETYPE = {
    "refresh_priority": "refresh_content",
    "ctr_fix_candidate": "fix_snippet_and_meta",
    "protect_and_expand": "leave_or_expand_thin_sections",
    "low_priority_deep": "low_priority_review",
    "monitor_only": "monitor_no_action",
}
df["action"] = df["archetype"].map(ACTION_BY_ARCHETYPE)

REASON_BY_ARCHETYPE = {
    "refresh_priority": "high_risk_visible_stale",
    "ctr_fix_candidate": "ctr_gap_visible_fresh",
    "protect_and_expand": "visible_fresh_ctr_ok",  # archetype is defined by freshness/CTR, not risk_score --
    "low_priority_deep": "low_value_deep_position",  # a row here can still carry a high model risk_score
    "monitor_only": "low_signal_not_visible",
}
df["reason_code"] = df["archetype"].map(REASON_BY_ARCHETYPE)

df["risk_score"] = risk_score
df["value_score"] = value_score
# Priority: risk-weighted value, but pages the model can't score reliably (monitor_only) sink
priority = np.where(df["archetype"] == "monitor_only", 0.0, risk_score * np.log1p(value_score))
df["playbook_priority"] = priority
df["playbook_rank"] = df["playbook_priority"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("playbook_rank")[[
    "playbook_rank", "content_id", "client_id", "archetype", "action", "reason_code",
    "risk_score", "value_score", "playbook_priority", "position_tier", "freshness_tier",
    "ctr", "impressions_90d", "clicks_90d", "cpc", "days_since_last_update",
]].reset_index(drop=True)

print("Archetype distribution:")
print(df["archetype"].value_counts())
print("\nTop 10 of the ranked queue:")
print(queue.head(10).to_string(index=False))


Archetype distribution:
archetype
monitor_only          13274
refresh_priority       6575
ctr_fix_candidate      4723
protect_and_expand     4006
low_priority_deep      1422
Name: count, dtype: int64

Top 10 of the ranked queue:
 playbook_rank           content_id         client_id          archetype                        action             reason_code  risk_score  value_score  playbook_priority position_tier freshness_tier  ctr  impressions_90d  clicks_90d   cpc  days_since_last_update
             1 content_68573149bb42 client_19581e27de  ctr_fix_candidate          fix_snippet_and_meta   ctr_gap_visible_fresh    0.648313       488.62           4.015412        page_1           0-30 0.12            18502          22 22.21                      20
             2 content_72e800a9c214 client_3fdba35f04   refresh_priority               refresh_content high_risk_visible_stale    0.765498       177.12           3.967160        page_1         91-180 0.12            13790          16 11.07    

## 2. Intended use and limits

**Intended use:** a weekly triage list for a content team lead -- "review these N pages this
sprint, in this order, for this reason." It is **decision-support**, not an autonomous system:
every row still needs a human to open the page and confirm the diagnosis before anything changes.

**Where it stops being valid:**
- **One portfolio, one snapshot.** Built on 30,000 rows from this dataset's 90-day window.
  Numbers are observed/measured for this data, not a universal SEO law (per
  `writing-honest-claims`'s claim ladder -- no causal language here).
- **Cross-sectional, not causal.** The model ranks by association, not by "doing X will cause
  Y." A `refresh_priority` label means "refreshing pages like this was associated with better
  outcomes in this data," not a guarantee for any single page.
- **Client-grouped validation, small held-out set.** precision@50 = 0.78 came from 6 held-out
  clients (~2,325 rows) -- a real number, but a small one. A very different client (new
  industry, new market) may not fit the pattern this model learned.
- **The model doesn't know *why*.** It flags risk, not root cause. A `refresh_priority` page
  might actually have a technical/indexing problem a content refresh won't fix (see the
  near-zero-CTR finding from `w05_model.ipynb`'s error analysis).
- **Value proxy, not booked revenue.** `clicks x cpc` is a click-equivalent-value estimate
  (the paper's own Finding #9 framing), not confirmed dollars.


In [2]:
# No computation needed -- this section documents intended use in plain language.
# Quick honesty check: print the exact validated number this playbook leans on, so the claim
# above stays traceable to a real artifact rather than a remembered figure.
print("Validated precision@50 this playbook's risk_score relies on (from w06_validation_audit.ipynb,")
print("client-grouped split, Random Forest): 0.78 on 2,325 held-out rows from 6 held-out clients.")
print("Base rate in that held-out set: ~0.54 (see w05_model.ipynb / w06 outputs).")


Validated precision@50 this playbook's risk_score relies on (from w06_validation_audit.ipynb,
client-grouped split, Random Forest): 0.78 on 2,325 held-out rows from 6 held-out clients.
Base rate in that held-out set: ~0.54 (see w05_model.ipynb / w06 outputs).


## 3. Human review + the no-go list

**What a human must check before acting on any row:**
1. Open the page and confirm the diagnosis matches reality (e.g., a `refresh_priority` page
   really does look outdated, not just old by the date field).
2. For `ctr_fix_candidate` rows, rule out a technical cause first (indexing, canonical,
   cannibalization) before rewriting a snippet -- per the Week-5 error analysis, near-zero-CTR
   pages in good positions are often a technical issue, not a content one.
3. Check whether the page has already been refreshed recently outside this dataset's window
   (the 90-day window can lag a real-world fix).
4. Confirm the page isn't YMYL (health, legal, financial) content where any change needs
   subject-matter and compliance review regardless of the model's score.

**Should NEVER be automated:**
- Auto-publishing rewritten content, titles, or meta descriptions without a human approving
  the specific wording.
- Auto-deleting, unpublishing, redirecting, or noindexing any page ("zombie" cleanup is a
  human decision with business/legal implications, not a script's).
- Any client-facing communication ("your content is declining") generated directly from the
  score -- risk scores are internal triage signals, not verified client-ready claims.
- Bulk actions triggered purely by `playbook_rank` crossing a threshold, with no per-page look.
- Treating `monitor_only` rows as "safe" -- it means "not enough signal yet," not "healthy."


In [3]:
# No computation needed -- policy text above. Quick sanity print: how many rows in each
# archetype would even be ELIGIBLE for the highest-stakes action (refresh_content), to show
# the no-go list isn't guarding an empty set.
print(df.loc[df["action"] == "refresh_content", "archetype"].value_counts())
print(f"\n{(df['action']=='refresh_content').sum():,} rows are currently flagged refresh_content --")
print("every one of them requires the human-review steps above before any change ships.")


archetype
refresh_priority    6575
Name: count, dtype: int64

6,575 rows are currently flagged refresh_content --
every one of them requires the human-review steps above before any change ships.


## 4. Monitoring / retrain triggers

**What would tell us the recommendations went stale:**
- **Performance drift:** recompute precision@50 on a fresh client-grouped held-out slice
  monthly. If it drops meaningfully below the validated 0.78 (e.g., below the Week-4 rule
  baseline's 0.66), pause automated ranking and fall back to the simpler rule-based baseline
  until investigated.
- **Data drift:** track the monthly distribution of `position_tier`, `freshness_tier`, and
  `impression_tier` mix (the paper's own trend table shows the portfolio's active-content count
  nearly doubled some months -- a queue trained on one mix can misread a very different one).
  A large shift (e.g., >20% relative change in any tier's share) is a retrain trigger.
- **Label staleness:** `trend_direction` is a rolling 30-day comparison -- refresh the training
  data at least monthly so the model isn't learning from a stale trend window.
- **Archetype collapse:** if `monitor_only` share grows sharply (more of the portfolio becoming
  too low-visibility to score), that's a coverage problem worth flagging even if precision looks fine.
- **Retrain cadence:** quarterly by default, or immediately if a performance-drift or
  data-drift trigger above fires early.


In [4]:
# Concrete numbers behind the monitoring triggers above, computed from this run.
current_mix = df["position_tier"].value_counts(normalize=True).round(3)
print("Current position_tier mix (compare future runs against this baseline):")
print(current_mix)

monitor_share = (df["archetype"] == "monitor_only").mean()
print(f"\nCurrent monitor_only share: {monitor_share:.1%} -- track this over time; a sharp rise")
print("means coverage is shrinking even if the scored rows still look accurate.")


Current position_tier mix (compare future runs against this baseline):
position_tier
page_1      0.394
striking    0.243
page_3_5    0.241
top_3       0.077
deep        0.044
Name: proportion, dtype: float64

Current monitor_only share: 44.2% -- track this over time; a sharp rise
means coverage is shrinking even if the scored rows still look accurate.


## 5. Exports for the paper

Ranked queue (full, all reason codes/actions) goes to `work/outputs/` -- regenerated on every
run, stays out of git by design. Archetype distribution figure and a metrics-summary JSON
(the receipts) are committed, since the paper's recommendations section builds directly on them.


In [5]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- CSV export (gitignored, regenerated every run) ---
out_path = Path("../outputs/content_action_playbook.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows to {out_path}")

# --- Figure export (committed) ---
fig_dir = Path("../figures")
fig_dir.mkdir(parents=True, exist_ok=True)

counts = df["archetype"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(counts.index, counts.values, color="#2f6f4f")
ax.set_xlabel("Number of pages")
ax.set_title("Content action playbook: pages by archetype")
for i, v in enumerate(counts.values):
    ax.text(v, i, f"  {v:,}", va="center")
plt.tight_layout()
fig_path = fig_dir / "archetype_distribution.png"
plt.savefig(fig_path, dpi=120)
plt.close(fig)
print(f"Saved figure to {fig_path}")

# --- Metrics JSON export (committed -- the receipts) ---
metrics = {
    "validated_precision_at_50_client_grouped": 0.78,
    "validated_precision_at_20_client_grouped": 0.90,
    "held_out_clients": 6,
    "held_out_rows": 2325,
    "held_out_base_rate": round(float(y.mean()), 3),
    "total_rows_scored": int(len(df)),
    "archetype_counts": df["archetype"].value_counts().to_dict(),
    "action_counts": df["action"].value_counts().to_dict(),
    "monitor_only_share": round(float(monitor_share), 4),
    "position_tier_mix": current_mix.to_dict(),
}
metrics_path = Path("../outputs/w07_playbook_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics receipts to {metrics_path}")


Wrote 30,000 rows to ../outputs/content_action_playbook.csv


Saved figure to ../figures/archetype_distribution.png
Saved metrics receipts to ../outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.